# 🕵️ Ontology Detective — Case File

> Build stamp: **2026-06-05 18:29:08** · *Datapolis P.I.*

Five cases. One ontology. Your reputation on the line.

**Loop for every case**
1. Read the **briefing**.
2. Open **`DetectiveOntology`** in Fabric **Digital Twin Builder** UI; add the entity
   types and relationships the briefing calls for.
3. Write a **KQL query** against the case tables until it returns a single suspect.
4. Call `detective.accuse(case_id, "Name")`.

Solve all five to be promoted **Commissioner of Datapolis** and mint your badge.

## ⚙️ Step 0 — Player

In [ ]:
# --- EDIT THIS ---
PLAYER_NAME = "Your Name Here"   # shown on your shareable badge
# -----------------
print(f"Player: {PLAYER_NAME}")

## Step 1 — Endpoints, helpers, judge

In [ ]:
import os, uuid, json, time, datetime as dt, requests
from IPython.display import Markdown, display, HTML

EH_NAME = "Datapolis_DetectiveEH"
DB_NAME = "Datapolis_DetectiveEH"

try:
    import notebookutils
    WORKSPACE_ID = notebookutils.runtime.context.get("currentWorkspaceId")
    _gettoken    = notebookutils.credentials.getToken
except Exception:
    import mssparkutils
    WORKSPACE_ID = mssparkutils.runtime.context.get("currentWorkspaceId")
    _gettoken    = mssparkutils.credentials.getToken

FAB = "https://api.fabric.microsoft.com/v1"
SESSION_ID = str(uuid.uuid4())
PLAYER_ID  = os.environ.get("USER", "detective")

def _fab(url):
    r = requests.get(url, headers={"Authorization": f"Bearer {_gettoken('pbi')}"}, timeout=60)
    r.raise_for_status(); return r.json()

dbs = _fab(f"{FAB}/workspaces/{WORKSPACE_ID}/items?type=KQLDatabase").get("value", [])
target = next((d for d in dbs if d["displayName"] == DB_NAME), None)
if not target: raise RuntimeError(f"KQL DB '{DB_NAME}' not found")
DB_ID = target["id"]
KQL_URI = _fab(f"{FAB}/workspaces/{WORKSPACE_ID}/kqlDatabases/{DB_ID}")["properties"]["queryServiceUri"]
print("Player    :", PLAYER_NAME)
print("Session   :", SESSION_ID)
print("Endpoint  :", KQL_URI)

In [ ]:
def query_kql(csl: str):
    '''Run a KQL query against Datapolis_DetectiveEH. Returns list[dict].'''
    tok = _gettoken("kusto")
    r = requests.post(f"{KQL_URI}/v2/rest/query",
                      headers={"Authorization": f"Bearer {tok}", "Content-Type": "application/json"},
                      json={"csl": csl, "db": DB_NAME}, timeout=120)
    if r.status_code != 200:
        raise RuntimeError(f"KQL query {r.status_code}: {r.text[:500]}")
    # v2 response: locate the PrimaryResult frame
    for frame in r.json():
        if frame.get("FrameType") == "DataTable" and frame.get("TableKind") == "PrimaryResult":
            cols = [c["ColumnName"] for c in frame["Columns"]]
            return [dict(zip(cols, row)) for row in frame["Rows"]]
    return []

def _mgmt(csl: str):
    tok = _gettoken("kusto")
    r = requests.post(f"{KQL_URI}/v1/rest/mgmt",
                      headers={"Authorization": f"Bearer {tok}", "Content-Type": "application/json"},
                      json={"csl": csl, "db": DB_NAME}, timeout=120)
    if r.status_code != 200:
        raise RuntimeError(f"KQL mgmt {r.status_code}: {r.text[:500]}")

def _q(v):
    s = str(v).replace("'", "\\'")
    return f"'{s}'"

def log_event(event_type: str, case_id: str = "", accused: str = "",
              result: str = "INFO", duration_s: int = 0, rank: str = ""):
    cmd = (
        f".ingest inline into table DetectiveEvents <|\n"
        f"{uuid.uuid4()},{dt.datetime.utcnow().isoformat()}Z,{SESSION_ID},{PLAYER_ID},"
        f"{event_type},{case_id},{accused},{result},{duration_s},{rank}"
    )
    try: _mgmt(cmd)
    except Exception as e: print(f"(telemetry suppressed: {e})")

## Step 2 — Detective class & ranks

In [ ]:
# The truth (no encryption — fair game; the goal is learning, not security theatre).
CULPRITS = {
    "stolen-pie":      "Bob Hollowstone",
    "museum":          "Lady Marlowe",
    "phone-call":      "Vincenzo Lupara",
    "stolen-identity": "Ricardo Vega",
    "final-heist":     "Madame Cinquedeo",
}

CASES = [
    ("stolen-pie",      "🧁 The Stolen Pie"),
    ("museum",          "🏛️ Disappearance at the Museum"),
    ("phone-call",      "📞 The Mysterious Phone Call"),
    ("stolen-identity", "🎭 Stolen Identity"),
    ("final-heist",     "🌃 The Final Heist (BOSS)"),
]

RANKS = [
    (0, "Aspiring Detective"),
    (1, "🥉 Rookie"),
    (2, "🥈 Investigator"),
    (3, "🎯 Inspector"),
    (4, "🔍 Senior Detective"),
    (5, "🏆 Commissioner of Datapolis"),
]
def rank_for(solved: int) -> str:
    out = RANKS[0][1]
    for n, r in RANKS:
        if solved >= n: out = r
    return out

class Detective:
    def help(self):
        md = ["### 🕵️ Detective Sherlock Graph — Datapolis P.I.\n",
              "Methods:",
              "- `detective.briefing(case_id)` — re-print a case briefing",
              "- `detective.case_list()` — show all 5 cases + status",
              "- `detective.accuse(case_id, \"Name\")` — submit your accusation",
              "- `detective.scoreboard()` — cases solved, rank, accuracy"]
        display(Markdown("\n".join(md)))

    def case_list(self):
        solved = self._solved_set()
        rows = ["| # | Case | Status |", "|---|------|--------|"]
        for i, (cid, name) in enumerate(CASES, start=1):
            tag = "✅ Solved" if cid in solved else "🔓 Open"
            rows.append(f"| {i} | {name} | {tag} |")
        display(Markdown("\n".join(rows)))

    def briefing(self, case_id: str):
        md = BRIEFINGS_MD.get(case_id)
        if not md:
            print(f"❌ Unknown case: {case_id}. Try one of: {list(BRIEFINGS_MD)}")
            return
        log_event("CaseOpened", case_id=case_id, result="INFO")
        display(Markdown(md))

    def _solved_set(self) -> set:
        try:
            rows = query_kql(
                f"DetectiveEvents | where SessionId == '{SESSION_ID}' "
                f"and EventType == 'CaseSolved' | summarize by CaseId"
            )
            return {r["CaseId"] for r in rows if r.get("CaseId")}
        except Exception:
            return set()

    def accuse(self, case_id: str, accused: str):
        if case_id not in CULPRITS:
            print(f"❌ Unknown case: {case_id}"); return
        expected = CULPRITS[case_id]
        log_event("AccusationMade", case_id=case_id, accused=accused, result="INFO")
        if accused.strip().lower() == expected.lower():
            log_event("CaseSolved", case_id=case_id, accused=accused, result="CORRECT")
            display(Markdown(
                f"### ✅ Case `{case_id}` — solved\n\n"
                f"**{accused}** is in the holding cell. Datapolis sleeps easier."
            ))
        else:
            log_event("WrongAccusation", case_id=case_id, accused=accused, result="WRONG")
            display(Markdown(
                f"### ❌ Wrong accusation\n\n"
                f"**{accused}** has been released with apologies. Re-read the briefing, "
                f"rework your KQL, and try again with `detective.accuse(\"{case_id}\", ...)`."
            ))
        self.scoreboard()

    def scoreboard(self):
        try:
            acc = query_kql(
                f"DetectiveEvents | where SessionId == '{SESSION_ID}' "
                f"and EventType in ('CaseSolved','WrongAccusation') "
                f"| summarize Solved=countif(EventType=='CaseSolved'), "
                f"Wrong=countif(EventType=='WrongAccusation')"
            )
        except Exception as e:
            print(f"(scoreboard unavailable: {e})"); return
        row = acc[0] if acc else {"Solved": 0, "Wrong": 0}
        solved = int(row.get("Solved", 0) or 0)
        wrong  = int(row.get("Wrong",  0) or 0)
        rank   = rank_for(solved)
        display(Markdown(
            f"### 🏛️ Detective scoreboard (this session)\n\n"
            f"- Cases solved : **{solved}/5**\n"
            f"- Wrong calls  : **{wrong}**\n"
            f"- Rank         : **{rank}**\n\n"
            f"> Solve all 5 cases, then run the badge cell at the bottom of this notebook."
        ))
        globals()["FINAL_SCORE"] = solved
        globals()["FINAL_RANK"]  = rank

detective = Detective()
log_event("DetectiveOnDuty", result="INFO")
print("🕵️ Detective on duty. Try: detective.help()")

## Step 3 — Briefings (markdown)

In [ ]:
BRIEFINGS_MD = {'stolen-pie': '## 🧁 Case #1 — The Stolen Pie\n\n*Case ID: `stolen-pie`*\n\n\n> *Rain on the windows. The phone rings before I\'ve finished my second coffee. Mrs. Plum,\n> bless her, has lost her championship blueberry pie. Cooling on the kitchen counter, 14:00.\n> Back from the garden at 14:30 — gone. Three neighbours dropped by that afternoon. One of\n> them left with a warm pie and a guilty smile.*\n\n**The scene** — Mrs. Plum\'s kitchen, **14:00 to 14:30**, Datapolis suburbs.\n\n**Suspects** — Bob Hollowstone · Alice Greengrass · Mortimer Quill *(plus Mrs. Plum herself)*.\n\n**Evidence** — every entry into a room of the house was logged by Mrs. Plum\'s smart-home\ncamera. The table `Case1_Visits` has columns `PersonName`, `RoomName`, `EnteredAt`, `LeftAt`.\n\n**Your job** — build an ontology with at minimum `Person`, `Room`, and a `Visit` relation,\nthen write a KQL query that returns the **single person, not Mrs. Plum, who was in the\nKitchen during the theft window**. Call `detective.accuse("<their name>")`.\n', 'museum': "## 🏛️ Case #2 — Disappearance at the Museum\n\n*Case ID: `museum`*\n\n\n> *Two million credits walked out of the Datapolis Museum last night. The Vector Vase —\n> Etruscan, fourth century BC, behind glass that should've held. The gala was packed.\n> Eight VIPs, two hundred lesser guests, and every camera in the house humming. The\n> alarm tripped at **21:14**. The vase was already gone by **21:18**.*\n\n**The scene** — Etruscan Hall, theft window **21:14 → 21:18**, gala evening 2026-06-04.\n\n**Evidence** — `Case2_CameraEvents` (`GuestName`, `Location`, `SeenAt`). The cameras\nsampled each guest as they entered a new room.\n\n**Your job** — extend the ontology with `Location` and a temporal `CameraEvent`. Find the\n**one guest** captured at the Etruscan Hall **between 21:14 and 21:18 inclusive**. Accuse\nthem.\n", 'phone-call': "## 📞 Case #3 — The Mysterious Phone Call\n\n*Case ID: `phone-call`*\n\n\n> *Senator Carballo dropped at dinner last night. Coroner says aconitine — a poison so\n> rare in Datapolis only one chemist still keeps it: Doc Aconite. We pulled the phone\n> records. The killer didn't poison Carballo himself — he had the dose delivered. Means\n> he called Carballo to know when, and called Doc Aconite to get the stuff. Within a\n> two-hour window.*\n\n**The scene** — phone records for 2026-06-03, `Case3_PhoneCalls` (`Caller`, `Callee`,\n`CalledAt`).\n\n**Your job** — model `Person` with a self-relationship `called`. Find the **one person\nwho placed a call to BOTH `Senator Carballo` AND `Dr. Aconite`, with the two calls less\nthan 2 hours apart**. Hint: `datetime_diff('minute', t1, t2)`. Accuse them.\n", 'stolen-identity': '## 🎭 Case #4 — Stolen Identity\n\n*Case ID: `stolen-identity`*\n\n\n> *Three men in Datapolis answer to "Mr. Vega." One\'s a real estate broker. One\'s a\n> teenage influencer. One\'s a forger working out of a back room in Quantum Hall. A\n> stolen-card report from Hotel Quantum, night of June 1st: someone checked in under a\n> Vega alias and ran up 12k in charges before vanishing. The receipt\'s signed in a name\n> that isn\'t on any ID. But every Vega has aliases.*\n\n**The scene** — Hotel Quantum, check-in window **22:00 on 2026-06-01 → 02:00 on 2026-06-02**.\n\n**Evidence** — `Case4_Aliases` (`AliasName`, `RealName`) maps every known alias to the\nreal Vega. `Case4_HotelCheckIns` (`UsedName`, `HotelName`, `CheckedInAt`) is the\nfront-desk register.\n\n**Your job** — model `Person` + an `Alias` with a `sameAs` link. **Resolve** the alias of\nthe check-in at Hotel Quantum in the window, then accuse the **real Vega** behind it.\n', 'final-heist': '## 🌃 Case #5 — The Final Heist (BOSS)\n\n*Case ID: `final-heist`*\n\n\n> *Fifty million crypto-bonds out of the Central Bank vault at exactly 23:05 last night.\n> Whoever pulled this off needed three things lined up: a clean shell account to land the\n> bonds in, a way into the Bank District without showing up on the patrol register, and a\n> burner-line call placed in the ten-minute window of the heist. Three agencies — Bank,\n> Police, Telecom — each gave us a partial list. One name is on all three.*\n\n**The scene** — 2026-06-05, heist window **23:00 → 23:10**, Bank District.\n\n**Evidence (3 sources)**\n- `Case5_BankAccounts` — who opened an `AccountKind == "RelayShell"`\n- `Case5_PolicePatrols` — who was seen in `PatrolZone == "Bank District"` with\n  `OnDutyRegister == false`\n- `Case5_BurnerCalls` — who called `Callee == "+39-X-USA-E-GETTA"` in the heist window\n\n**Your job** — federate three sub-namespaces (`Bank.Account`, `Police.Patrol`,\n`Telecom.PhoneCall`) under one shared `Person`. Compute the **intersection**. Accuse the\nsingle name that appears in all three. *(Note: this case is on you to solve. The whole\ncity is watching.)*\n'}

## Step 4 — Play

In [ ]:
detective.help()

In [ ]:
# detective.case_list()

In [ ]:
# detective.briefing("stolen-pie")

In [ ]:
# --- Example KQL workspace (run any case query here) ---
# rows = query_kql('''
# Case1_Visits
# | where RoomName == "Kitchen"
# | where EnteredAt < datetime(2026-06-05 14:30:00)
#   and LeftAt   > datetime(2026-06-05 14:00:00)
# | where PersonName != "Mrs. Plum"
# | project PersonName
# ''')
# rows

In [ ]:
# detective.accuse("stolen-pie", "Bob Hollowstone")

## Step 5 — Show all briefings + scoreboard

Run this after you've worked each case. It re-prints the 5 briefings (handy as a quick recap) and shows your current scoreboard / rank. Then move on to the badge cell.

In [ ]:
for cid, _name in CASES:
    detective.briefing(cid)
detective.scoreboard()

## Step 6 — 🏅 Mint your shareable badge

In [ ]:
# ============================================================
# Ontology Detective — Badge issuance (same pattern as retro-arcade / city-builder)
# ============================================================
import json, time, hmac, hashlib, base64
from IPython.display import display, Markdown, HTML

_BADGE_SECRET = b"fabric-arcade-badge-v1-7K9mP3xQ"
_BASE_URL     = "https://maenglar78.github.io/fabric-arcade"
_GAME_ID      = "ontology-detective"
_SKILLS       = ["Fabric Ontology", "Digital Twin Builder", "KQL",
                 "Knowledge Graph", "Entity Resolution"]

def _b64u(b: bytes) -> str:
    return base64.urlsafe_b64encode(b).rstrip(b"=").decode("ascii")

def _issue(game_id, player, rank, score):
    payload = {"v": 1, "g": game_id, "p": str(player),
               "r": str(rank), "s": int(score), "t": int(time.time()),
               "k": _SKILLS}
    body = json.dumps(payload, separators=(",", ":"), sort_keys=True).encode()
    sig  = hmac.new(_BADGE_SECRET, body, hashlib.sha256).digest()
    return f"{_BASE_URL}/badge.html?t={_b64u(body)}.{_b64u(sig)}"

score = globals().get("FINAL_SCORE", 0)
rank  = globals().get("FINAL_RANK", "Aspiring Detective")

if score < 1:
    display(Markdown(
        f"### 🚧 Not yet eligible (cases solved {score}/5)\n\n"
        f"Solve **at least 1 case** to earn the Rookie badge. "
        f"Run `detective.accuse(...)` on any open case, then re-run "
        f"`detective.scoreboard()` and this cell."
    ))
elif PLAYER_NAME.strip() in ("", "Your Name Here"):
    display(Markdown(
        "### ✍️ Set your name first\n\n"
        "Edit `PLAYER_NAME` in **Step 0** and re-run `detective.scoreboard()` + this cell."
    ))
else:
    url = _issue(_GAME_ID, PLAYER_NAME, rank, score)
    display(Markdown(
        f"### 🏅 Badge minted\n\n"
        f"**{PLAYER_NAME}** — *{rank}* · cases solved **{score}/5**\n\n"
        f"🔗 **[Open your badge]({url})**\n\n"
        f"Click *Download PNG* / *Share on LinkedIn* on the badge page."
    ))
    display(HTML(f'<a href="{url}" target="_blank" '
                 f'style="display:inline-block;padding:10px 20px;border-radius:8px;'
                 f'background:linear-gradient(135deg,#00d4ff,#8338ec);color:white;'
                 f'text-decoration:none;font-weight:600">🏅 Open my badge page</a>'))
    log_event("BadgeIssued", case_id="all", accused=url, result="ISSUED", rank=rank)